In [49]:
import napari
import numpy as np
from aicsimageio import AICSImage

import pandas as pd
from skimage import measure


import tifffile as tf
import matplotlib.pyplot as plt
import xarray as xr
from tifffile import tifffile
import tifftools
import os
from matplotlib import pyplot as plt
from os.path import sep
from skimage import io
from PIL import Image
import imageio


from skimage.measure import regionprops_table

import seaborn as sns

import czifile

import math

from pathlib import Path


In [3]:
# Load the .czi file

fly3 = '/Users/fisherguest/Downloads/sansa images/03222025/SC-0322-fly3.czi'
fly4 = '/Users/fisherguest/Downloads/sansa images/03222025/SC-0322-fly4.czi'

fly5 = '/Users/fisherguest/Downloads/sansa images/03272025/SC-0327-fly5.czi'
fly6 = '/Users/fisherguest/Downloads/sansa images/03272025/SC-0327-fly6.czi'
fly7 = '/Users/fisherguest/Downloads/sansa images/03272025/SC-0327-fly7.czi'

fly8 = '/Users/fisherguest/Downloads/sansa images/04022025/SC-0402-fly8.czi'
fly9 = '/Users/fisherguest/Downloads/sansa images/04022025/SC-0402-fly9.czi'

fly10 = '/Users/fisherguest/Downloads/sansa images/04082025/SC-0408-fly10.czi'

fly11 = '/Users/fisherguest/Downloads/sansa images/04092025/SC-0409-fly11.czi'
fly12 = '/Users/fisherguest/Downloads/sansa images/04092025/SC-0409-fly12.czi'
fly13 = '/Users/fisherguest/Downloads/sansa images/04092025/SC-0409-fly13.czi'

fly1 = '/Users/fisherguest/Downloads/sansa images/03172025/SC-0317-fly1.czi'

In [40]:
current_fly = fly1

In [41]:
# import image:
img = AICSImage(current_fly)

# Get the data in (C, Z, Y, X)
# C: Channels (e.g., the different lasers/fluorophores); Z: Z-stacks (slices in depth); Y and X: The spatial dimensions (height and width)
# i don't have S or T.
data = img.get_image_data("CZYX", S=0, T=0)

# checked using "previous TQ method": 568 is first channel, 488 is second channel.
red_channel   = data[0]
green_channel = data[1]


In [42]:
# open image with set contrast limits (so don't darg the bar):
viewer = napari.Viewer()

viewer.add_image(
    red_channel, 
    name='568 Channel', 
    colormap='red',
    blending='additive',
    contrast_limits=(100, 2500)  # if right-click on the contrast-limits bar, will see this upper and lower boundary; if not set, will be the original values.
) # so to find the best limits range, open the original image first to see the 

viewer.add_image(
    green_channel, 
    name='488 Channel', 
    colormap='green',
    blending='additive',
    contrast_limits=(50, 800)   # specify a different range for the green channel
)

napari.run()

In [43]:
# File paths
label_file = '/Users/fisherguest/Downloads/sansa images/03172025/roi/fly1-18roi.tif'


# Load the label mask (the TIFF file you saved from napari)
label_mask = tf.imread(label_file)
# for the label tif file obtained in previous TQ method, need the below precessing:
#squeezed_mask = np.squeeze(label_mask) # so change from shape: (1, 2, Z, Y, X) to (2, Z, Y, X)
#squeezed_mask_green = squeezed_mask[1] # so only extract the green (second) channel.
print("Label mask shape:", label_mask.shape)
print("Green channel shape:", green_channel.shape)

# Check that the dimensions match (e.g., both are 3D)
if green_channel.shape != label_mask.shape:
    raise ValueError("The dimensions of the green channel and the label mask do not match. "
                     "Please verify that your label mask corresponds to the correct z, y, and x dimensions.")

Label mask shape: (298, 1839, 1839)
Green channel shape: (298, 1839, 1839)


In [44]:
# check which z stack I drew the label on.
sums = label_mask.sum(axis=(1,2))
painted_slices = np.where(sums > 0)[0]
print("You painted on slice(s):", painted_slices)

You painted on slice(s): [177]


In [45]:
z_ranges = [
(124, 188),
(115, 193),
(125, 220),
(142, 220),
(154, 222),
(152, 222),
(169, 271),
(216, 285),
(219, 287),
(184, 282),
(160, 270),
(152, 274),
(144, 217),
(141, 221),
(104, 203),
(92, 173),
(93, 167),
(91, 170),
]

In [46]:
# variable input cell
red_threshold_lower = 200
green_threshold_lower = 0
green_threshold_higher = 3000


In [47]:
n_glom    = int(label_mask.max())  # should be 18
mean_red   = np.zeros(n_glom, dtype=float)
mean_green = np.zeros(n_glom, dtype=float)
total_reds   = np.zeros(n_glom, dtype=float)
total_greens = np.zeros(n_glom, dtype=float)

for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)  # THIS COMMENT WILL SOLVE MANY POTENTIAL QUESTIONS: label_mask size is in this format: (Z, Y, X)
    z_draw = zs[0]                          # AICSImage makes the image format in this way! (i don't know why not in X->Y->Z)

    # 2) extract the 2D ROI on that slice:
    roi2d = (label_mask[z_draw] == i)   # label_mask[z_draw] gives 2d array (Y, X): entries are the integer labels (background:0, ROIs:1-18) 
                                        # label_mask[z_draw] == i: give each pixel True of False based on if it's has label 1.

    # 3) build a 1D mask for your Z‑range:
    z0, z1   = z_ranges[i-1]
    Z, Y, X  = label_mask.shape         # label_mask is the full X * full Y * full Z
    mask_z   = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)  # np.arange(Z): generates the array [0, 1, 2, …, Z−1]
                                                            # np.arange(Z) >= z0: generates the array [False, F, ...T, T, ... F]
                                                            # () & (): generates the array with True only for indices between z0 and z1.

    # 4) make the final 3d mask:
    roi = mask_z[:, None, None] & roi2d[None, :, :]         # mask_z[:, None, None]: mask_z is originally 1d array, but using None it adds on dimension with length of 1
                                                                # so now it's a 3d array with the first dimension (or first element) length of z-stack range, the second and the third length of 1
                                                            # mask_z[:, None, None] size: (z-stack range, 1, 1)
                                                            # roi2d[None, :, :] size: (1, 1863, 1863) with some x and some y True.
                                                            # Then, the "&" step: for example: 
                                                                    # mz goes from (3,1,1) → (3,2,2) by repeating the single 1×1 mask across the 2×2 grid in X and Y.
                                                                    # r2 goes from (1,2,2) → (3,2,2) by repeating the same 2×2 slice across the 3 Z‑layers.
                                                            # in this new 3d array: first dimension (element) is Z!
                                                            # so: in this new 3d array:
                                                                    # first look at mask_z, if it is true, that means there is label in this z-stack,
                                                                        # and thus i will allow the Y and X info from roi2d to get copied into this Z dimension.
                                                                    # if it's False, then there is no label on this z-stack,
                                                                        # so i won't allow roi2d to copy anything to here, and this whole Z dimension will have all False values
            # for example: mask_z = np.array([ True, False,  True ]), roi2d  = np.array([[ True, False], [False,  True]]),
                # then roi = array([[[ True, False], [False,  True]],   # z=0 slice uses roi2d because mask_z[0] is True
                #                   [[False, False], [False, False]],   # z=1 slice is all False because mask_z[1] is False
                #                   [[ True, False], [False,  True]]])  # z=2 slice repeats roi2d because mask_z[2] is True.
    
    # 5) mask out everything outside ROI in your channels
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]      # red_channel[z0:z1+1]: takes only the z-stacks from the original image (red channel only) that are within the current label z-stack range
                                                           # roi[z0:z1+1]: only slicing in the first (Z) dimension!
                                                           # after multiplication: it's a smaller Z * 1863 * 1863 3d array, with the in-label pixels having value of 1*it's original value
    green_sub = green_channel[z0:z1+1]                     # only select out the correct z stacks, haven't picked the inner pixels yet!

    # 6) build combined mask: red > red_threshold_lower AND green > green_threshold_lower
    mask = (red_sub > red_threshold_lower) & (green_sub > green_threshold_lower) & (green_sub < green_threshold_higher)    # after making the correct label in each z stack, "mask" is the mask for correct pixels.

    # 7) extract means only from voxels passing both thresholds
    red_vals   = red_sub  [mask]                           # only extract the value from the pixel in "mask"
    green_vals = green_sub[mask]

    mean_red[i-1]   = red_vals.mean()   if red_vals.size   else np.nan
    mean_green[i-1] = green_vals.mean() if green_vals.size else np.nan
    total_reds[i-1]   = red_vals.sum()
    total_greens[i-1] = green_vals.sum()

In [48]:
labels = [f"glom_{i}" for i in range(1, n_glom+1)]
df_means = pd.DataFrame({
    'mean_red':   mean_red,
    'mean_green': mean_green,
    'total_red':   total_reds,
    'total_green': total_greens
}, index=labels)
df_means.index.name = 'glomerulus'

# 2) Derive the “fly3” shortname from your CZI path
#    fly3 = '/…/SC-0322-fly3.czi'
base      = os.path.basename(current_fly)          # 'SC-0322-fly3.czi'
stem      = os.path.splitext(base)[0]       # 'SC-0322-fly3'
shortname = stem.split('-')[-1]             # 'fly3'
csv_filename = f"{shortname}_mean_signals.csv"

# 3) Display and save
display(df_means)                           # Jupyter display
df_means.to_csv(csv_filename)

print(f"Saved results to: {csv_filename}")

,mean_red,mean_green,total_red,total_green
glomerulus,,,,
glom_1,558.327650,132.828103,1.466291e+08,34883582.0
glom_2,581.856527,130.795287,3.136736e+08,70510562.0
glom_3,571.637483,130.483335,2.470303e+08,56387721.0
glom_4,518.672671,123.303412,2.680443e+08,63721847.0
glom_5,510.587536,100.598360,1.822358e+08,35904963.0
glom_6,497.076504,114.629280,1.709262e+08,39416768.0
glom_7,860.203080,173.452017,1.204585e+09,242893358.0
glom_8,450.353637,77.809039,1.262476e+08,21812208.0
glom_9,435.602818,84.157248,8.201922e+07,15845884.0


Saved results to: fly1_mean_signals.csv


In [ ]:
# check the mask but also filter voxel >100 in green channel

# Prepare storage and (optionally) propagated_labels as before…
propagated_labels = np.zeros_like(label_mask, dtype=np.int32)

for i in range(1, n_glom+1):
    # 1) find where you drew the label
    zs, ys, xs = np.where(label_mask == i)
    z_draw = zs[0]

    # 2) 2D ROI on drawn slice
    roi2d = (label_mask[z_draw] == i)

    # 3) Z‑range mask
    z0, z1  = z_ranges[i-1]
    Z, Y, X = label_mask.shape
    mask_z  = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)

    # 4) propagate to 3D
    roi = mask_z[:, None, None] & roi2d[None, :, :]

    # 5) mask out everything outside ROI in your channels
    red_sub   = red_channel  [z0:z1+1] * roi[z0:z1+1]
    green_sub = green_channel[z0:z1+1]         

    # 6) build combined mask: red > red_threshold_lower AND green > green_threshold_lower
    mask = (red_sub   > red_threshold_lower) & \
           (green_sub > green_threshold_lower)

    # 7) extract means only from voxels passing both thresholds
    red_vals   = red_sub  [mask]
    green_vals = green_sub[mask]

    mean_red[i-1]   = red_vals.mean()   if red_vals.size   else np.nan
    mean_green[i-1] = green_vals.mean() if green_vals.size else np.nan

    # 8) (optional) paint propagated_labels for Napari
    propagated_labels[z0:z1+1][mask] = i

#Save the new propagated label volume to TIFF
tf.imwrite('propagated_glom_labels.tif', propagated_labels.astype(np.uint16))

# Now open it in Napari alongside your images
viewer = napari.Viewer()
viewer.add_image(red_channel,   name='568 nm', colormap='red',   blending='additive')
viewer.add_image(green_channel, name='488 nm', colormap='green', blending='additive')
viewer.add_labels(propagated_labels, name='Propagated ROIs', opacity=0.6)
napari.run()

In [ ]:
# drafts to just check understanding:

In [24]:
zs, ys, xs = np.where(label_mask == 1)
z_draw = zs[0]        # the only slice you drew on

# 2) extract the 2D ROI on that slice:
roi2d = (label_mask[z_draw] == 1)  
roi2d

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [25]:
label_mask[z_draw][41]

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [26]:
has_true = roi2d.any()         # returns True if at least one element is True
print(has_true)

True


In [27]:
Z, Y, X  = label_mask.shape
Z

119

In [28]:
a = np.array([ True, False,  True ])
a.shape

(3,)

In [29]:
z0, z1   = z_ranges[1-1]
Z, Y, X  = label_mask.shape
mask_z   = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)  # shape (Z,)

# 4) broadcast to 3D:
#    True only where (a) Z in [z0,z1] and (b) XY
roi = mask_z[:, None, None] & roi2d[None, :, :] 

In [31]:
mask_z

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False])

In [32]:
roi.shape

(119, 1839, 1839)

In [33]:
# extra sanity check:

In [41]:
mask = roi  # your boolean array, shape (119, 1839, 1839)

# Method A: np.nonzero
zs, ys, xs = np.nonzero(mask)
# Now (zs[k], ys[k], xs[k]) is the k-th True‐voxel.

zs[5]

16

In [51]:
a = np.array([ True, False,  True, False ])
b = np.array([[ True, False], [False,  True], [False,  True]])
c = a[:, None, None] & b[None, :, :]

d = np.array([[[ 1, 2],[3,  4], [5,  6]],
              [[7, 8],[ 9, 10], [11, 12]],
              [[13, 14], [15,  16], [17, 18]],
              [[19, 20], [21,  22], [23, 24]]
             ])
e   = d[0:0+1]   * c[0:0+1]
e

array([[[1, 0],
        [0, 4],
        [0, 6]]])

In [55]:
f = (e > 2)
f

array([[[False, False],
        [False,  True],
        [False,  True]]])

In [57]:
g = e[f]
g.mean()

5.0

In [35]:
z = 59
y0, y1 = 1815, 1838
x0, x1 = 1815, 1838

# Collect rows: (location, has_label1)
rows = []
for y in range(y0, y1+1):
    for x in range(x0, x1+1):
        loc = (z, y, x)
        has_label1 = (label_mask[z, y, x] == 1)
        rows.append({
            'location': loc,
            'has_label1': bool(has_label1)
        })

# Build DataFrame
df = pd.DataFrame(rows, columns=['location', 'has_label1'])
print(df)

             location  has_label1
0    (59, 1815, 1815)       False
1    (59, 1815, 1816)       False
2    (59, 1815, 1817)       False
3    (59, 1815, 1818)       False
4    (59, 1815, 1819)       False
..                ...         ...
571  (59, 1838, 1834)        True
572  (59, 1838, 1835)        True
573  (59, 1838, 1836)        True
574  (59, 1838, 1837)        True
575  (59, 1838, 1838)        True

[576 rows x 2 columns]


In [11]:
n_glom    = int(label_mask.max())  # should be 18
mean_red   = np.zeros(n_glom, dtype=float)
mean_green = np.zeros(n_glom, dtype=float)

for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)  # THIS COMMENT WILL SOLVE MANY POTENTIAL QUESTIONS: label_mask size is in this format: (Z, Y, X)
    z_draw = zs[0]                          # AICSImage makes the image format in this way! (i don't know why not in X->Y->Z)

    # 2) extract the 2D ROI on that slice:
    roi2d = (label_mask[z_draw] == i)   # label_mask[z_draw] gives 2d array (Y, X): entries are the integer labels (background:0, ROIs:1-18) 
                                        # label_mask[z_draw] == i: give each pixel True of False based on if it's has label 1.

    # 3) build a 1D mask for your Z‑range:
    z0, z1   = z_ranges[i-1]
    Z, Y, X  = label_mask.shape         # label_mask is the full X * full Y * full Z
    mask_z   = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)  # np.arange(Z): generates the array [0, 1, 2, …, Z−1]
                                                            # np.arange(Z) >= z0: generates the array [False, F, ...T, T, ... F]
                                                            # () & (): generates the array with True only for indices between z0 and z1.

    # 4) make the final 3d mask:
    roi = mask_z[:, None, None] & roi2d[None, :, :]         # mask_z[:, None, None]: mask_z is originally 1d array, but using None it adds on dimension with length of 1
                                                                # so now it's a 3d array with the first dimension (or first element) length of z-stack range, the second and the third length of 1
                                                            # mask_z[:, None, None] size: (z-stack range, 1, 1)
                                                            # roi2d[None, :, :] size: (1, 1863, 1863) with some x and some y True.
                                                            # Then, the "&" step: for example: 
                                                                    # mz goes from (3,1,1) → (3,2,2) by repeating the single 1×1 mask across the 2×2 grid in X and Y.
                                                                    # r2 goes from (1,2,2) → (3,2,2) by repeating the same 2×2 slice across the 3 Z‑layers.
                                                            # in this new 3d array: first dimension (element) is Z!
                                                            # so: in this new 3d array:
                                                                    # first look at mask_z, if it is true, that means there is label in this z-stack,
                                                                        # and thus i will allow the Y and X info from roi2d to get copied into this Z dimension.
                                                                    # if it's False, then there is no label on this z-stack,
                                                                        # so i won't allow roi2d to copy anything to here, and this whole Z dimension will have all False values
            # for example: mask_z = np.array([ True, False,  True ]), roi2d  = np.array([[ True, False], [False,  True]]),
                # then roi = array([[[ True, False], [False,  True]],   # z=0 slice uses roi2d because mask_z[0] is True
                #                   [[False, False], [False, False]],   # z=1 slice is all False because mask_z[1] is False
                #                   [[ True, False], [False,  True]]])  # z=2 slice repeats roi2d because mask_z[2] is True.
    
    # — b) zero out outside your propagated ROI —
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]      # red_channel[z0:z1+1]: takes only the z-stacks from the original image (red channel only) that are within the current label z-stack range
                                                           # roi[z0:z1+1]: only slicing in the first (Z) dimension!
                                                           # after multiplication: it's a smaller Z * 1863 * 1863 3d array, with the in-label pixels having value of 1*it's original value
    green_sub = green_channel[z0:z1+1]                     # only select out the correct z stacks, haven't picked the inner pixels yet!

    # — c) threshold & extract means —
    mask      = (red_sub > red_threshold_lower)            # after making the correct label in each z stack, "mask" is the mask for correct pixels.
    red_vals   = red_sub  [mask]                           # only extract the value from the pixel in "mask"
    green_vals = green_sub[mask]

    mean_red[i-1]   = red_vals.mean()   if red_vals.size   else np.nan
    mean_green[i-1] = green_vals.mean() if green_vals.size else np.nan

In [12]:
labels = [f"glom_{i}" for i in range(1, n_glom+1)]
df_means = pd.DataFrame(
    [mean_red, mean_green],
    index=['mean_red', 'mean_green'],
    columns=labels
)
print("\nMean intensities per glomerulus:")
print(df_means)


Mean intensities per glomerulus:
                glom_1      glom_2
mean_red    774.666674  555.373648
mean_green   94.288838   47.688099


In [58]:
# check the mask in napari

propagated_labels = np.zeros_like(label_mask, dtype=np.int32)  # shape (Z, Y, X)

# Your existing loop, with one extra line to “paint” into propagated_labels
for i in range(1, n_glom+1):
    # 1) find the slice you painted on:
    zs, ys, xs = np.where(label_mask == i)
    z_draw = zs[0]

    # 2) 2D ROI on drawn slice
    roi2d = (label_mask[z_draw] == i)

    # 3) Z‑range mask
    z0, z1 = z_ranges[i-1]
    Z, Y, X = label_mask.shape
    mask_z = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)

    # 4) broadcast to 3D
    roi = mask_z[:, None, None] & roi2d[None, :, :]

    # 5) your threshold step and mean extraction
    red_sub   = red_channel[z0:z1+1]   * roi[z0:z1+1]
    mask_thr  = (red_sub > red_threshold_lower)
    # … compute mean_red[i-1], mean_green[i-1] as before …

    # — NEW: paint this ROI into your propagated label map —
    propagated_labels[roi & (red_channel > red_threshold_lower)] = i

# Save the new propagated label volume to TIFF
tf.imwrite('propagated_glom_labels.tif', propagated_labels.astype(np.uint16))

# Now open it in Napari alongside your images
viewer = napari.Viewer()
viewer.add_image(red_channel,   name='568 nm', colormap='red',   blending='additive')
viewer.add_image(green_channel, name='488 nm', colormap='green', blending='additive')
viewer.add_labels(propagated_labels, name='Propagated ROIs', opacity=0.6)
napari.run()

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [13]:
# check the mask but also filter voxel >100 in green channel

# Prepare storage and (optionally) propagated_labels as before…
propagated_labels = np.zeros_like(label_mask, dtype=np.int32)

for i in range(1, n_glom+1):
    # 1) find where you drew the label
    zs, ys, xs = np.where(label_mask == i)
    z_draw = zs[0]

    # 2) 2D ROI on drawn slice
    roi2d = (label_mask[z_draw] == i)

    # 3) Z‑range mask
    z0, z1  = z_ranges[i-1]
    Z, Y, X = label_mask.shape
    mask_z  = (np.arange(Z) >= z0) & (np.arange(Z) <= z1)

    # 4) propagate to 3D
    roi = mask_z[:, None, None] & roi2d[None, :, :]

    # 5) mask out everything outside ROI in your channels
    red_sub   = red_channel  [z0:z1+1] * roi[z0:z1+1]
    green_sub = green_channel[z0:z1+1]         

    # 6) build combined mask: red > red_threshold_lower AND green > green_threshold_lower
    mask = (red_sub   > red_threshold_lower) & \
           (green_sub > green_threshold_lower)

    # 7) extract means only from voxels passing both thresholds
    red_vals   = red_sub  [mask]
    green_vals = green_sub[mask]

    mean_red[i-1]   = red_vals.mean()   if red_vals.size   else np.nan
    mean_green[i-1] = green_vals.mean() if green_vals.size else np.nan

    # 8) (optional) paint propagated_labels for Napari
    propagated_labels[z0:z1+1][mask] = i

#Save the new propagated label volume to TIFF
tf.imwrite('propagated_glom_labels.tif', propagated_labels.astype(np.uint16))

# Now open it in Napari alongside your images
viewer = napari.Viewer()
viewer.add_image(red_channel,   name='568 nm', colormap='red',   blending='additive')
viewer.add_image(green_channel, name='488 nm', colormap='green', blending='additive')
viewer.add_labels(propagated_labels, name='Propagated ROIs', opacity=0.6)
napari.run()